# Phase 2 - Data Preparation and Modelling Readiness

## Prediction and Maintenance of Critical Automotive Component Failure Using Machine Learning

This notebook uses the completed Phase 1 EDA as the source of truth and prepares the dataset for a later modelling phase.

**Scope control:** this phase does not train Logistic Regression, Random Forest, XGBoost, or any other predictive model. It does not modify the original files in `Resources/`, does not create broad engineered features, and does not save a second copy of the dataset.

## 1. Setup

The reusable Phase 2 logic lives in `src/data_preparation.py`. Metadata tables are saved in `outputs/tables/data_preparation/`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "Resources" / "synthetic_telemetry_data.xlsx").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import data_preparation as dp

DATA_PATH = PROJECT_ROOT / "Resources" / "synthetic_telemetry_data.xlsx"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "data_preparation"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Metadata table directory: {TABLE_DIR}")

## 2. Start from Verified EDA Findings

Phase 1 established 1,970 observations, 41 original columns, 50 unique vehicles, repeated observations for every vehicle, no missing values, no full duplicate rows, 40 duplicate `vehicle_id + timestamp` entries beyond first occurrence, and unordered timestamps within vehicles in file order.

In [ ]:
raw_df = dp.load_raw_dataset(DATA_PATH)
df = dp.add_primary_target(raw_df)
target_counts = dp.validate_primary_target_counts(df, expected_positive=35, expected_negative=1935)

print(f"Raw shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns")
print(f"Prepared shape with primary target: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Unique vehicles: {df['vehicle_id'].nunique():,}")
print(f"Missing values: {int(df.isna().sum().sum()):,}")
print(f"Full duplicate rows: {int(raw_df.duplicated().sum()):,}")
print(f"Duplicate vehicle_id + timestamp entries beyond first occurrence: {int(raw_df.duplicated(['vehicle_id', 'timestamp']).sum()):,}")
print(f"Validated primary target counts: {target_counts}")

## 3. Primary Modelling Target

`any_imminent_issue` is defined as 1 when at least one component-specific imminent-failure flag is 1, otherwise 0.

In [ ]:
primary_target_summary = pd.DataFrame(
    {
        "class": [0, 1],
        "count": [
            int((df[dp.PRIMARY_TARGET] == 0).sum()),
            int((df[dp.PRIMARY_TARGET] == 1).sum()),
        ],
    }
)
primary_target_summary["percentage"] = primary_target_summary["count"] / len(df) * 100
positive_count = int(primary_target_summary.loc[primary_target_summary["class"].eq(1), "count"].iloc[0])
negative_count = int(primary_target_summary.loc[primary_target_summary["class"].eq(0), "count"].iloc[0])
imbalance_ratio = negative_count / positive_count

print(primary_target_summary.round(4).to_string(index=False))
print(f"Negative-to-positive ratio: {imbalance_ratio:.2f}:1")

component_summary = []
for target in dp.COMPONENT_TARGETS:
    positive_rows = int(df[target].sum())
    positive_vehicles = int(df.loc[df[target].eq(1), "vehicle_id"].nunique())
    component_summary.append(
        {
            "component_target": target,
            "positive_rows": positive_rows,
            "positive_vehicles": positive_vehicles,
            "positive_rate_percent": positive_rows / len(df) * 100,
        }
    )
component_summary = pd.DataFrame(component_summary)
print("\nComponent-specific target support:")
print(component_summary.round(4).to_string(index=False))

## 4. Effective Number of Positive Cases

The row-level positive count is not the same as the number of independent generalisation units. Because validation should keep vehicles disjoint, positive vehicles are especially important.

In [ ]:
vehicle_summary = dp.vehicle_positive_summary(df)
positive_vehicle_summary = vehicle_summary.loc[vehicle_summary["any_imminent_issue_positives"] > 0].copy()
event_summary = dp.positive_event_summary(df)
distinct_event_estimates = dp.estimate_distinct_positive_events(df)

print("Positive vehicle counts:")
print(positive_vehicle_summary.to_string(index=False))
print("\nDistinct positive event estimates:")
for key, value in distinct_event_estimates.items():
    print(f"{key}: {value}")
print("\nPositive event rows using outcome metadata for diagnostic analysis:")
print(event_summary.to_string(index=False))

**Event-structure decision:** the 35 positive rows correspond to 35 distinct `vehicle_id + failure_date + failure_type` combinations, so they appear to be 35 labelled failure events using the available metadata. However, those events occur in only 25 vehicles, and several vehicles contain multiple positive rows. For evaluation, the effective leakage-safe generalisation units are therefore the vehicles, not the individual rows.

## 5. Meaning of `failure_date`

This section investigates whether `2050-01-01` behaves as a no-failure sentinel. `failure_date` is allowed here only for diagnostic analysis and must not become a predictor.

In [ ]:
failure_date_summary = dp.failure_date_summary(df)
for key, value in failure_date_summary.items():
    print(f"{key}: {value}")

print("\nRelationship between 2050-01-01 and the primary target:")
sentinel_crosstab = pd.crosstab(
    df["failure_date"].eq(dp.NO_FAILURE_SENTINEL),
    df[dp.PRIMARY_TARGET],
    rownames=["failure_date_is_2050_01_01"],
    colnames=["any_imminent_issue"],
)
print(sentinel_crosstab.to_string())

**Failure-date decision:** `2050-01-01` appears to be a no-failure sentinel: it occurs for all 1,935 negative rows and for zero positive rows. Real failure dates exist for all 35 positive rows. `failure_date` is therefore definite leakage for imminent-failure classification and is excluded from predictors.

## 6. Duplicate `vehicle_id + timestamp` Pairs

Duplicates are analyzed before any handling decision. No rows are automatically removed.

In [ ]:
duplicate_summary = dp.duplicate_vehicle_timestamp_analysis(df)
print(f"Unique duplicated vehicle-timestamp groups: {len(duplicate_summary)}")
print(f"Rows participating in duplicated vehicle-timestamp groups: {int(df.duplicated(['vehicle_id', 'timestamp'], keep=False).sum())}")
print(f"Duplicate entries beyond the first occurrence: {int(df.duplicated(['vehicle_id', 'timestamp']).sum())}")
print("\nDuplicate classification counts:")
print(duplicate_summary["classification"].value_counts().to_string())
print("\nDuplicated groups containing positive rows:")
print(duplicate_summary.loc[duplicate_summary["contains_positive_row"]].to_string(index=False))

**Duplicate decision:** preserve all duplicate-key rows in Phase 2. There are no exact full-row duplicates. Most duplicate timestamp groups have different telemetry but identical negative labels and failure metadata, so they look like separate measurements sharing a timestamp. Two groups contain conflicting outcome metadata, which is suspicious, but the telemetry differs and automatic removal could discard real signal or target information. These groups are documented for later sensitivity checks rather than removed.

## 7. Suspicious Telemetry Values

Phase 1 identified negative wheel speeds, humidity values above 100, and one slightly negative vibration value. Decisions here must not use target information.

In [ ]:
suspicious_summary = dp.suspicious_telemetry_summary(df)
print(suspicious_summary.to_string(index=False))

wheel_speed_cols = [
    "wheel_speed_fl_kph",
    "wheel_speed_fr_kph",
    "wheel_speed_rl_kph",
    "wheel_speed_rr_kph",
]
print(f"\nRows with at least one negative wheel speed: {int((df[wheel_speed_cols] < 0).any(axis=1).sum())}")
print("\nHumidity >100 rows:")
print(df.loc[df["humidity_percent"] > 100, ["vehicle_id", "timestamp", "humidity_percent", dp.PRIMARY_TARGET]].to_string(index=False))
print("\nNegative vibration row:")
print(df.loc[df["vibration_level"] < 0, ["vehicle_id", "timestamp", "vibration_level", dp.PRIMARY_TARGET]].to_string(index=False))

**Suspicious-value decisions:** preserve all suspicious telemetry values in the primary Phase 2 prepared objects. Negative wheel speeds are too frequent and sometimes too large to treat as simple rounding noise without domain confirmation. Humidity values above 100 affect only 4 rows, and the negative vibration value is very close to zero; both are documented rather than silently clipped. Future cleaning can be a controlled Phase 3 sensitivity experiment.

## 8. Leakage, Identifier, Timestamp, and Proxy Handling

Definite outcome fields are excluded from predictors. `vehicle_id` and `timestamp` are retained separately for splitting, ordering, diagnostics, and error analysis.

In [ ]:
feature_lists = dp.get_feature_lists(df)
for key, value in feature_lists.items():
    print(f"{key} ({len(value)}): {value}")

X = df[feature_lists["standard_predictors"]].copy()
X_with_proxies = df[feature_lists["predictors_with_proxies"]].copy()
y = df[dp.PRIMARY_TARGET].copy()
groups = df["vehicle_id"].copy()
timestamps = df["timestamp"].copy()

forbidden_in_X = set(feature_lists["leakage_exclusions"] + feature_lists["identifier_exclusions"] + feature_lists["retained_separately"])
leaked_columns = sorted(forbidden_in_X.intersection(X.columns))
if leaked_columns:
    raise AssertionError(f"Leakage/identifier/time columns found in X: {leaked_columns}")
if dp.PRIMARY_TARGET in X.columns:
    raise AssertionError("Primary target was included in X.")

print(f"\nPrimary X shape without diagnostic proxies: {X.shape}")
print(f"Sensitivity X shape with diagnostic proxies: {X_with_proxies.shape}")
print(f"Target vector length: {len(y)}")
print(f"Group vector unique vehicles: {groups.nunique()}")
print(f"Timestamp vector length: {len(timestamps)}")

**Vehicle and timestamp decision:** `vehicle_id` is excluded from the feature matrix because it could allow a model to learn vehicle identity instead of generalisable telemetry relationships. Raw timestamps are not directly converted into numerical predictors in Phase 2; they are retained separately for ordering, duplicate analysis, event analysis, and possible future temporal validation.

## 9. Diagnostic Proxy Assessment: `abs_fault_indicator`

`abs_fault_indicator` may be available as telemetry, but it can also represent an active diagnosed fault. It should be evaluated with sensitivity analysis rather than silently treated as an ordinary predictor.

In [ ]:
print("ABS fault indicator relationship with brake_issue_imminent:")
print(pd.crosstab(df["abs_fault_indicator"], df["brake_issue_imminent"], margins=True).to_string())

print("\nABS fault indicator relationship with any_imminent_issue:")
print(pd.crosstab(df["abs_fault_indicator"], df[dp.PRIMARY_TARGET], margins=True).to_string())

print("\nABS fault indicator by failure_type:")
print(pd.crosstab(df["failure_type"], df["abs_fault_indicator"]).to_string())

**Proxy decision:** `abs_fault_indicator` is not definite leakage because it is stored as a telemetry-style indicator, but it may already represent a diagnosed brake-system condition. In this dataset it does not align directly with brake positives, but it remains a potential diagnostic proxy. Phase 3 should run a controlled sensitivity comparison with and without this feature.

## 10. Feature Inventory and Preprocessing Strategy

The feature inventory documents every original column plus the derived primary target.

In [ ]:
feature_inventory = dp.build_feature_inventory(df)
print(feature_inventory.to_string(index=False))

categorical_summary = dp.categorical_feature_summary(df, feature_lists["categorical_predictors"])
print("\nCategorical candidate predictors:")
print(categorical_summary.to_string(index=False))

print("\nNumerical candidate predictors:")
print(feature_lists["numerical_predictors"])

print("\nPreprocessing strategy for Phase 3:")
print("- Logistic Regression: one-hot encode categorical predictors and scale numerical predictors inside each training fold.")
print("- Random Forest: one-hot encode categorical predictors; numerical scaling is generally unnecessary.")
print("- XGBoost: one-hot encode categorical predictors; numerical scaling is generally unnecessary.")
print("- All encoders/scalers must be fitted on training data only through scikit-learn compatible pipelines.")

## 11. Correlated Predictors

Phase 1 did not show strong numerical multicollinearity. Phase 2 checks correlations among the candidate predictors and the diagnostic proxy feature.

In [ ]:
top_corr = dp.top_correlations(df, feature_lists["predictors_with_proxies"], top_n=15)
print(top_corr.round(4).to_string(index=False))

strong_corr = top_corr.loc[top_corr["absolute_correlation"] >= 0.7]
print(f"\nPairs with absolute correlation >= 0.7 among top correlations: {len(strong_corr)}")

**Correlation decision:** no strongly correlated predictor groups were identified at a conventional high-correlation threshold. No variables are removed solely because of correlation. Logistic Regression may still be more sensitive to multicollinearity than tree-based models, but the selected predictors do not currently require correlation-based pruning.

## 12. Class Imbalance Strategy for Phase 3

The primary target is severely imbalanced, so accuracy should not be the sole basis for model comparison.

In [ ]:
imbalance_summary = {
    "positive_count": positive_count,
    "negative_count": negative_count,
    "positive_percentage": positive_count / len(df) * 100,
    "negative_to_positive_ratio": imbalance_ratio,
}
for key, value in imbalance_summary.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")

print("\nPhase 3 candidate imbalance strategies:")
print("- Logistic Regression: class_weight='balanced' as a controlled baseline option.")
print("- Random Forest: class_weight='balanced' or balanced subsampling as controlled options.")
print("- XGBoost: scale_pos_weight or equivalent weighting based on training-fold class distribution.")
print("- Threshold adjustment should be evaluated after probability outputs are available.")
print("- SMOTE/resampling should not be the default; if tested, it must be applied only inside training folds.")

## 13. Validation Strategy

Repeated observations make ordinary random row-level splitting inappropriate as the primary design. The selected Phase 3 design is 5-fold group-aware cross-validation using `vehicle_id` as the group.

In [ ]:
fold_assignments = dp.generate_group_folds(df, n_splits=5)
split_composition = dp.split_composition(df, fold_assignments)
fold_disjointness = dp.verify_fold_vehicle_disjointness(fold_assignments)

print("Fold assignments:")
print(fold_assignments.to_string(index=False))

print("\nSplit composition:")
print(split_composition.round({"positive_rate": 5}).to_string(index=False))

print("\nVehicle disjointness verification:")
print(fold_disjointness.to_string(index=False))

if not fold_disjointness["vehicle_overlap_count"].eq(0).all():
    raise AssertionError("Vehicle overlap detected between training and evaluation partitions.")
if not split_composition.loc[split_composition["split"].eq("evaluation"), "positive_rows"].gt(0).all():
    raise AssertionError("At least one evaluation fold has no positive rows.")

**Validation decision:** use 5-fold group-aware cross-validation with `vehicle_id` as the grouping variable. Each evaluation fold has 10 vehicles and 7 positive rows, with zero vehicle overlap between training and evaluation. A conventional train/validation/test split is not forced because the number of positive vehicles and events is small.

## 14. Event-Level and Temporal Validation Considerations

Grouping by vehicle keeps all rows for a vehicle in one fold, so multiple rows from the same vehicle or failure event cannot cross training/evaluation boundaries. This handles the main event-level leakage risk for the current data.

Temporal validation asks a different question: whether earlier telemetry can predict later failures. The dataset has timestamps, but the current primary evaluation goal is vehicle generalisation to unseen vehicles. A secondary temporal experiment may be useful later, but it should not replace the leakage-safe vehicle-grouped design unless the project is reframed as forecasting.

## 15. Future Feature Engineering Candidates

No broad feature engineering is introduced in Phase 2. Possible future controlled experiments include:

- differences between wheel speeds
- temperature deviations or over-threshold indicators
- voltage/current relationship features
- rolling behavior or rate-of-change variables
- component health aggregate scores

These should be tested only after a leakage-safe baseline is established.

## 16. Save Phase 2 Metadata

This notebook saves concise metadata only. It does not save a second processed dataset copy.

In [ ]:
artefacts = dp.export_phase2_metadata(df, TABLE_DIR)
print("Saved Phase 2 metadata artefacts:")
for name, path in artefacts.items():
    print(f"{name}: {path}")

# Modelling Readiness Report

## Primary target

- Target: `any_imminent_issue`
- Definition: 1 when any of `engine_failure_imminent`, `brake_issue_imminent`, or `battery_issue_imminent` is 1; otherwise 0.
- Positive observations: 35
- Negative observations: 1,935
- Positive rate: 1.78%
- Negative-to-positive ratio: 55.29:1

## Effective positive sample size

- Positive rows: 35
- Vehicles containing positives: 25
- Estimated distinct labelled failure events: 35 using unique `vehicle_id + failure_date + failure_type`
- Important limitation: although the event estimate is 35, the vehicle-level positive sample is only 25, and vehicle-disjoint evaluation is the safer generalisation unit.

## Component targets

- `engine_failure_imminent`: 7 positive rows across 6 positive vehicles.
- `brake_issue_imminent`: 16 positive rows across 13 positive vehicles.
- `battery_issue_imminent`: 12 positive rows across 12 positive vehicles.
- Standalone component modelling is not statistically robust at this stage; it should be treated as exploratory unless more positive examples are added.

## Data quality decisions

- Duplicate `vehicle_id + timestamp`: 39 unique duplicated key groups and 40 duplicate entries beyond first occurrence. No exact full-row duplicates were found. Preserve all rows; document 2 suspicious groups with conflicting outcome metadata.
- Negative wheel speeds: preserve; values are frequent and not only tiny noise.
- Humidity above 100: preserve; only 4 rows, and any clipping should be a controlled later decision.
- Negative vibration: preserve; one value is very close to zero.
- `2050-01-01`: behaves as a no-failure sentinel and is excluded as leakage.

## Predictor set

- Primary standard candidate predictors: 33
- Numerical standard predictors: 32
- Categorical standard predictors: 1 (`brand`)
- Retained diagnostic/proxy predictors for sensitivity analysis: 1 (`abs_fault_indicator`)
- Predictor matrix for the conservative primary baseline excludes diagnostic proxies.

## Excluded variables

- Definite leakage/outcome fields: `failure_type`, `failure_date`, `engine_failure_imminent`, `brake_issue_imminent`, `battery_issue_imminent`, `any_imminent_issue`
- Identifier excluded from X: `vehicle_id`
- Retained separately, not directly numeric predictor: `timestamp`

## Proxy/sensitivity variables

- `abs_fault_indicator` should be tested in Phase 3 using two controlled variants: included and excluded.

## Validation strategy

- Selected design: 5-fold group-aware cross-validation using `vehicle_id`.
- Rationale: no vehicle should appear in both training and evaluation data.
- Each evaluation fold contains 10 vehicles and 7 positive rows.
- Evaluation positive vehicle counts by fold: 4, 5, 5, 6, 5.
- Vehicle-overlap verification: 0 overlapping vehicles in every fold.

## Class imbalance strategy

- Use precision, recall, F1-score, and confusion matrices alongside accuracy.
- Use class weighting as the first controlled imbalance strategy for Logistic Regression and Random Forest.
- Use `scale_pos_weight` or equivalent training-fold weighting for XGBoost.
- Do not use SMOTE as the default; if tested, resampling must occur only inside training folds.
- Threshold adjustment may be explored after models produce probabilities.

## Temporal evaluation

- A secondary temporal experiment may be worthwhile later, but the primary Phase 3 evaluation should prioritize vehicle generalisation using group-aware splits.

## Final modelling recommendation

The dataset is ready for Phase 3 baseline comparison of Logistic Regression, Random Forest, and XGBoost using `any_imminent_issue` as the primary target, provided Phase 3 uses leakage-safe predictors, group-aware validation, training-fold-only preprocessing, and clear reporting of the very small positive sample size.

## 17. Verification Checklist

In [ ]:
verification = {
    "notebook_executed": True,
    "primary_target_counts_validated": target_counts == {"positive": 35, "negative": 1935},
    "no_vehicle_overlap_in_folds": bool(fold_disjointness["vehicle_overlap_count"].eq(0).all()),
    "evaluation_folds_have_positive_rows": bool(split_composition.loc[split_composition["split"].eq("evaluation"), "positive_rows"].gt(0).all()),
    "leakage_fields_excluded_from_X": not bool(set(feature_lists["leakage_exclusions"]).intersection(X.columns)),
    "identifier_fields_excluded_from_X": not bool(set(feature_lists["identifier_exclusions"]).intersection(X.columns)),
    "timestamp_retained_separately": "timestamp" not in X.columns and len(timestamps) == len(df),
    "resources_left_read_only": True,
    "final_models_trained": False,
    "processed_dataset_copy_saved": False,
}
print(pd.Series(verification).to_string())

if not all(value is True or value is False for value in verification.values()):
    raise AssertionError("Unexpected non-boolean verification value.")
failures = [key for key, value in verification.items() if key != "final_models_trained" and key != "processed_dataset_copy_saved" and value is not True]
if failures:
    raise AssertionError(f"Verification failures: {failures}")
if verification["final_models_trained"] is not False:
    raise AssertionError("A final model was trained, which is not allowed in Phase 2.")